# Attempting to Model NGC6569 with PyfalcON


In [ ]:
import numpy as np

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation

import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 200  # Set to 100MB or whatever you need

from IPython.display import HTML

import pandas as pd

import pyfalcon

In [ ]:
import astropy.coordinates as coord
import astropy.units as u
from astropy.constants import G
import gala.coordinates as gc
import gala.dynamics as gd
import gala.potential as gp
from gala.units import galactic
from gala.dynamics import mockstream as ms

import agama

import importlib
import sys

from time import time

from leap_frog import kdk_leapfrog_TD
from leap_frog import kdk_leapfrog

#from leap_frog_hdf5 import kdk_leapfrog_hdf5, load_simulation_hdf5

In [ ]:
# default Astropy Galactocentric frame parameters to the values adopted in Astropy v4.0:
_ = coord.galactocentric_frame_defaults.set('v4.0')

# set Agama units 
# working units: 1 Msun, 1 kpc, 1 km/s
agama.setUnits(length=1*u.kpc, velocity=1*u.km/u.s, mass=1*u.Msun)
print("Newton G in Agama units,",agama.G)

# Check the current unit system
print("Current Agama units:")
print(f"Length unit: {agama.getUnits()['length']}")
print(f"Velocity unit: {agama.getUnits()['velocity']}")  
print(f"Time unit: {agama.getUnits()['time']}")
print(f"Mass unit: {agama.getUnits()['mass']}")

agama_time_unit = agama.getUnits()["time"]
print(agama_time_unit)

### NEW: binary output helpers

Small binary-table helpers used in the new cells only. The data go to raw float64 `.bin` files, with one small JSON sidecar per file so the shape and columns can be read back without guessing.


In [ ]:
from pathlib import Path
import json
import hashlib

OUT = Path("output/dev3_original_binary_1gyr")
OUT.mkdir(parents=True, exist_ok=True)

def write_manifest(path, manifest):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(manifest, indent=2))
    return path

def write_binary_table(path, data, columns, units=None, description=""):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    arr = np.asarray(data, dtype=np.float64)
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    arr.tofile(path)
    meta = {
        "binary_file": path.name,
        "dtype": "float64",
        "shape": list(arr.shape),
        "columns": list(columns),
        "units": units or {},
        "description": description,
    }
    write_manifest(path.with_suffix(".json"), meta)
    return path

def read_binary_table(path):
    path = Path(path)
    meta = json.loads(path.with_suffix(".json").read_text())
    data = np.fromfile(path, dtype=np.dtype(meta["dtype"])).reshape(meta["shape"])
    return data, meta

def binary_table_ready(path):
    path = Path(path)
    return path.exists() and path.with_suffix(".json").exists()

def binary_cache_key(payload):
    return hashlib.md5(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:10]


In [ ]:
# Use the Hunter rotating potential 
pot_ext = agama.Potential("milkyway/MWPotentialHunter24_full.ini") 
pot_rot = agama.Potential("milkyway/MWPotentialHunter24_rotating.ini") 
pot_bovy = agama.Potential("milkyway/MWPotential2014.ini") 

pot_use = pot_rot

In [ ]:
# NG6569 coordinates 

c = coord.SkyCoord(ra = 273.412*u.degree, dec = -31.827*u.degree,
                        distance=(10.5)*u.kpc,
                        pm_ra_cosdec= -4.125*u.mas/u.yr,
                        pm_dec= -7.354*u.mas/u.yr,
                        radial_velocity= -49.82*u.km/u.s)

# transform to galactic centeric 
c_gc = c.transform_to(coord.Galactocentric).data
print(c_gc._differentials)

In [ ]:
# creat phase space object
w0 = gd.PhaseSpacePosition(c_gc)
print("initial position :", w0.pos)
print("initial velocity :", w0.vel)
print("Need to convert velocity to km/s")

pos_0 = np.r_[w0.pos.x.value, w0.pos.y.value, w0.pos.z.value]
vel_0 = np.r_[w0.vel.d_x.to(u.km/u.s).value, 
              w0.vel.d_y.to(u.km/u.s).value,
              w0.vel.d_z.to(u.km/u.s).value]
print("position", pos_0) 
print("velocity", vel_0) 

In [ ]:
# integrate orbit 
tfin= -1000*u.Myr  # NEW: 1 Gyr run; original Dev3 used -200*u.Myr
nt=2000
t_eval = np.linspace(0, tfin, nt)
t_scipy = t_eval.to(u.Gyr).value/agama_time_unit.to(u.Gyr).value
t_scipy

In [ ]:
def rhs(t,state): 
    
    pos = state[:3]
    vel = state[3:]

    acc = pot_use.force(pos, t=t)
    return np.r_[vel, acc] 

# integrate with solve_ivp
from scipy.integrate import solve_ivp

state_0 = np.r_[pos_0, vel_0] 
print(state_0, state_0.shape)

t_span = (0, t_scipy[-1]) 
sol = solve_ivp(rhs, t_span, state_0, t_eval =t_scipy, rtol=1e-10, atol=1e-10)
orbit = sol["y"]

In [ ]:
x = orbit[0,:]
y = orbit[1,:]
z = orbit[2,:]
vx = orbit[3,:]
vy = orbit[4,:]
vz = orbit[5,:]

In [ ]:
# Agama orbit
# Calculate orbit
orbit = agama.orbit(potential=pot_use, 
                   ic=state_0, 
                   time=t_span[1],      # Total integration time
                   trajsize=nt)   # Number of output points

In [ ]:
# 2-D orbit figures 
fig = plt.figure(figsize=(14, 4))

ax = fig.add_subplot(131)
ax.set_xlabel("x [kpc]", size=20) 
ax.set_ylabel("y [kpc]", size=20)
ax.plot(x,y, label="from scratch")
ax.plot(orbit[1][:,0], orbit[1][:,1], "--", label="Agama") 
ax.legend()

ax = fig.add_subplot(132)
ax.set_xlabel("x [kpc]", size=20) 
ax.set_ylabel("z [kpc]", size=20)
ax.plot(x,z)
ax.plot(orbit[1][:,0], orbit[1][:,2], "--") 

ax = fig.add_subplot(133)
ax.set_xlabel("y [kpc]", size=20) 
ax.set_ylabel("z [kpc]", size=20)
ax.plot(y,z)
ax.plot(orbit[1][:,1], orbit[1][:,2], "--") 
plt.tight_layout()
fig.savefig("ngc6569_oribit.pdf")

In [ ]:
# Define the parameters for your King model
W0_value = 7.0  # Example W0 value
# create an isolated star cluster
r_scale = 1/1000
m = 2.3*1e5*(2)
pot_sat = agama.Potential(type='king', W0=W0_value, scaleRadius=r_scale, mass=m)
df_sat = agama.DistributionFunction(type='quasispherical', potential=pot_sat)
Nbody = 150000
xv, mass = agama.GalaxyModel(pot_sat, df_sat).sample(Nbody)

r_agama = np.sqrt(xv[:,0]**2 + xv[:,1]**2 + xv[:,2]**2) 
v_agama = np.sqrt(xv[:,3]**2 + xv[:,4]**2 + xv[:,5]**2) 

print("Agama G:", agama.G)

cluster_data = np.c_[mass, xv]
cluster_data.shape
# OLD text output:
# np.savetxt("cluster_data.txt", cluster_data)

### NEW: binary save for the original King sample

This replaces the old `cluster_data.txt` text output while leaving the original King-model setup above intact.


In [ ]:
write_binary_table(
    OUT / "cluster_data.bin",
    cluster_data,
    columns=["mass", "x", "y", "z", "vx", "vy", "vz"],
    units={"mass": "Msun", "x": "kpc", "y": "kpc", "z": "kpc", "vx": "km/s", "vy": "km/s", "vz": "km/s"},
    description="Original Dev3 King-model sample saved as a raw binary table.",
)


### NEW: King sampler function and distribution checks

The King sampler lives in `gc_initial_conditions.py`; this cell imports it and runs the parameter checks. The grid below changes W0 and the scale radius, then saves binary tables and histogram PDFs so the parameter effects are easy to compare.


In [ ]:
from gc_initial_conditions import sample_king_model

king_W0_values = [4.0, 7.0, 10.0]
king_scale_radii = [0.5/1000, 1.0/1000, 2.0/1000]
king_Nbody_demo = 10000
king_mass_demo = m

fig_r, axes_r = plt.subplots(len(king_W0_values), len(king_scale_radii), figsize=(11, 8), sharex=False, sharey=True)
fig_v, axes_v = plt.subplots(len(king_W0_values), len(king_scale_radii), figsize=(11, 8), sharex=False, sharey=True)

king_grid_rows = []
king_particle_rows = []

for i, W0_demo in enumerate(king_W0_values):
    for j, r_scale_demo in enumerate(king_scale_radii):
        xv_demo, mass_demo, _, _ = sample_king_model(W0_demo, r_scale_demo, king_mass_demo, king_Nbody_demo)
        r_demo = np.linalg.norm(xv_demo[:, :3], axis=1)
        v_demo = np.linalg.norm(xv_demo[:, 3:], axis=1)

        axes_r[i, j].hist(r_demo, bins=60, histtype="step", color="black")
        axes_r[i, j].set_title(f"W0={W0_demo:g}, rc={r_scale_demo:.4f} kpc", fontsize=9)
        axes_r[i, j].set_xlabel("r [kpc]")
        axes_v[i, j].hist(v_demo, bins=60, histtype="step", color="black")
        axes_v[i, j].set_title(f"W0={W0_demo:g}, rc={r_scale_demo:.4f} kpc", fontsize=9)
        axes_v[i, j].set_xlabel("speed [km/s]")

        model_id = i * len(king_scale_radii) + j
        king_grid_rows.append([
            model_id, W0_demo, r_scale_demo, king_Nbody_demo, np.sum(mass_demo),
            np.median(r_demo), np.percentile(r_demo, 90), np.median(v_demo), np.percentile(v_demo, 90),
        ])
        king_particle_rows.append(np.column_stack((
            np.full(king_Nbody_demo, model_id),
            np.full(king_Nbody_demo, W0_demo),
            np.full(king_Nbody_demo, r_scale_demo),
            mass_demo,
            xv_demo,
            r_demo,
            v_demo,
        )))

for ax in axes_r[:, 0]:
    ax.set_ylabel("count")
for ax in axes_v[:, 0]:
    ax.set_ylabel("count")
fig_r.suptitle("King radial distributions")
fig_v.suptitle("King speed distributions")
fig_r.tight_layout()
fig_v.tight_layout()
fig_r.savefig(OUT / "king_radial_histograms.pdf")
fig_v.savefig(OUT / "king_speed_histograms.pdf")

king_grid_table = np.asarray(king_grid_rows)
king_particles_table = np.vstack(king_particle_rows)

write_binary_table(
    OUT / "king_parameter_grid.bin",
    king_grid_table,
    columns=["model_id", "W0", "scale_radius", "Nbody", "total_mass", "median_r", "r90", "median_speed", "speed90"],
    units={"scale_radius": "kpc", "total_mass": "Msun", "median_r": "kpc", "r90": "kpc", "median_speed": "km/s", "speed90": "km/s"},
    description="Summary statistics for the King W0/scale-radius grid.",
)
write_binary_table(
    OUT / "king_particles.bin",
    king_particles_table,
    columns=["model_id", "W0", "scale_radius", "mass", "x", "y", "z", "vx", "vy", "vz", "r", "speed"],
    units={"scale_radius": "kpc", "mass": "Msun", "x": "kpc", "y": "kpc", "z": "kpc", "vx": "km/s", "vy": "km/s", "vz": "km/s", "r": "kpc", "speed": "km/s"},
    description="Particles sampled for the King W0/scale-radius histogram grid.",
)


In [ ]:
fig = plt.figure(figsize=(14,6))

ax=fig.add_subplot(121)
ax.set_xlabel("radial distance", size=20) 
ax.hist(r_agama, bins=30, edgecolor="black", alpha=.5, label ="Agama")

ax.axvline(r_scale, color="black", label="core radius")
ax.legend(loc=0) 

ax=fig.add_subplot(122)
ax.set_xlabel("speed", size=20) 
ax.hist(v_agama, bins=30, edgecolor="black", alpha=.5, label ="Agama")

ax.legend(loc=0) 


In [ ]:
pos_0 = np.r_[x[-1], y[-1], z[-1]]
vel_0 = np.r_[vx[-1], vy[-1], vz[-1]]  # NEW: use vz for velocity
print(pos_0)
print(vel_0)

center_pos_0 = pos_0.copy()
center_vel_0 = vel_0.copy()


In [ ]:
def xv_to_xyz_vel(xv, center_pos, center_vel):
    """Convert King-model phase-space offsets to xyz and velocity arrays.

    Parameters
    ----------
    xv : array, shape (N, 6)
        King-model particle offsets: x, y, z, vx, vy, vz.
    center_pos : array, shape (3,)
        Galactocentric xyz position of the cluster center [kpc].
    center_vel : array, shape (3,)
        Galactocentric velocity of the cluster center [km/s].
    """
    xyz = xv[:, :3] + center_pos
    velo = xv[:, 3:] + center_vel

    print("checks")
    print(np.mean(xyz[:, 0]), np.mean(xyz[:, 1]), np.mean(xyz[:, 2]))
    print(center_pos)
    print(np.mean(velo[:, 0]), np.mean(velo[:, 1]), np.mean(velo[:, 2]))
    print(center_vel)

    return xyz, velo


In [ ]:
# pos_0 = np.column_stack((x, y, z))
# vel_0 = np.column_stack((vx, vy, vz))
# vel_0.shape

pos_0 , vel_0 = xv_to_xyz_vel(xv, center_pos_0, center_vel_0)

In [ ]:
np.sum(mass/1e5)
# OLD text output:
# np.savetxt("ngc6569_mass.txt", mass)

### NEW: binary save for NGC6569 particle masses

This replaces the old `ngc6569_mass.txt` text output.


In [ ]:
write_binary_table(
    OUT / "ngc6569_mass.bin",
    mass,
    columns=["mass"],
    units={"mass": "Msun"},
    description="Initial particle masses for the NGC6569 collisionless run.",
)


In [ ]:
time_unit=u.kpc.to(u.km)*u.s.to(u.Gyr)
time_unit

In [ ]:
tmax = -tfin.to(u.Gyr).value/time_unit
print("maximum time", tmax)
print(t_scipy[-1])

In [ ]:
kmax=16
tau = 2**(-kmax)*time_unit
print("time step:", tau) 
nt=int(tmax/tau) + 1
print("number of time steps:", nt)

eps = 1/1000  # II 
eps = .1/1000 # I 
eps_power = -4
eps = (2**eps_power)/1000
print("softening length:", eps) 

In [ ]:
downsample=20
#filename="ngc_6569_runI"
#nt=1000

### NEW: binary cache for the NGC6569 run

This follows the Chen-validation Step 2 idea: make a parameter hash, check for finished products, and skip the expensive integration when the binary files already match the current setup.


In [ ]:
USE_BINARY_CACHE = True

ngc6569_cache_payload = {
    "W0": float(W0_value),
    "scale_radius_kpc": float(r_scale),
    "cluster_mass_Msun": float(m),
    "Nbody": int(Nbody),
    "tfin_Myr": float(tfin.to(u.Myr).value),
    "kmax": int(kmax),
    "eps_kpc": float(eps),
    "downsample": int(downsample),
    "potential": "MWPotentialHunter24_rotating.ini",
}
ngc6569_cache_key = binary_cache_key(ngc6569_cache_payload)
ngc6569_required_cache_tables = [
    OUT / "ngc6569_initial_cluster_frame.bin",
    OUT / "ngc6569_petar_initial.bin",
    OUT / "ngc6569_final_particles.bin",
    OUT / "ngc6569_mass_loss.bin",
    OUT / "ngc6569_center_orbit.bin",
]

loaded_from_cache = False
manifest_path = OUT / "ngc6569_run_manifest.json"
if USE_BINARY_CACHE and manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    cache_ok = manifest.get("cache_key") == ngc6569_cache_key
    cache_ok = cache_ok and all(binary_table_ready(path) for path in ngc6569_required_cache_tables)
    if cache_ok:
        initial_cluster_table, _ = read_binary_table(OUT / "ngc6569_initial_cluster_frame.bin")
        initial_gc_table, _ = read_binary_table(OUT / "ngc6569_petar_initial.bin")
        final_gc_table, _ = read_binary_table(OUT / "ngc6569_final_particles.bin")
        mass_loss_table, _ = read_binary_table(OUT / "ngc6569_mass_loss.bin")
        center_orbit_table, _ = read_binary_table(OUT / "ngc6569_center_orbit.bin")

        mass = initial_gc_table[:, 0]
        xv = initial_cluster_table[:, 1:7]
        r_agama = initial_cluster_table[:, 7]
        v_agama = initial_cluster_table[:, 8]
        cluster_data = np.c_[mass, xv]
        write_binary_table(
            OUT / "cluster_data.bin",
            cluster_data,
            columns=["mass", "x", "y", "z", "vx", "vy", "vz"],
            units={"mass": "Msun", "x": "kpc", "y": "kpc", "z": "kpc", "vx": "km/s", "vy": "km/s", "vz": "km/s"},
            description="Cached Dev3 King-model sample restored from the NGC6569 run cache.",
        )
        write_binary_table(
            OUT / "ngc6569_mass.bin",
            mass,
            columns=["mass"],
            units={"mass": "Msun"},
            description="Cached particle masses restored from the NGC6569 run cache.",
        )
        pos_0 = initial_gc_table[:, 1:4]
        vel_0 = initial_gc_table[:, 4:7]

        traj = {
            "mass": mass_loss_table[:, 2],
            "energy": mass_loss_table[:, 4],
            "time": mass_loss_table[:, 1],
            "pos": center_orbit_table[:, 2:5],
            "vel": center_orbit_table[:, 5:8],
            "distance": center_orbit_table[:, 8],
        }
        time_array = traj["time"] * 1000.0
        sim_data = [{"time": traj["time"][-1], "pos": final_gc_table[:, 1:4], "vel": final_gc_table[:, 4:7]}]
        loaded_from_cache = True
        print(f"Loaded cached NGC6569 run products: {ngc6569_cache_key}")

if not loaded_from_cache:
    print(f"No matching NGC6569 binary cache yet: {ngc6569_cache_key}")


In [ ]:
if not loaded_from_cache:
    t1 = time()
    sim_data = kdk_leapfrog(pot_use, pos_0, vel_0,
                            mass, nt, tau, agama.G, eps,
                            time_unit, downsample,last_snapshot=False)
    t2=time()
    print("run time", t2-t1, (t2-t1)/60)
else:
    print("Skipping kdk_leapfrog; loaded cached binary products.")


In [ ]:
# get time array
if not loaded_from_cache:
    time_array=[]
    for i in range(len(sim_data)):

        t = sim_data[i]["time"]
        time_array.append(t*1000)

    time_array=np.array(time_array)


In [ ]:
from bound_funcs import get_bound_particles

if not loaded_from_cache:
    bound_data = get_bound_particles(sim_data, mass)
else:
    bound_data = None


In [ ]:
def trajectories(bound_data, sim_data): 
    """
    Extract time evolution trajectories of bound cluster properties from simulation data.
    
    This function processes the output from bound particle tracking to create clean
    time series arrays for analysis and plotting of cluster evolution.
    
    Parameters:
    -----------
    bound_data : list of dict
        List of dictionaries containing bound particle data at each timestep.
        Each dictionary should contain keys: 'total mass', 'total energy', 'pos', 'vel'
    sim_data : list of dict
        List of dictionaries containing simulation data at each timestep.
        Each dictionary should contain key: 'time'
        
    Returns:
    --------
    dict : Dictionary containing trajectory arrays
        'mass' : array, shape (n,) - Total mass of bound particles vs time
        'energy' : array, shape (n,) - Total energy of bound particles vs time  
        'time' : array, shape (n,) - Time array
        'pos' : array, shape (n, 3) - Center of mass position vs time
        'vel' : array, shape (n, 3) - Center of mass velocity vs time

    """
    
    n = len(bound_data)
    
    # Validate input lengths match
    if len(sim_data) != n:
        raise ValueError(f"Length mismatch: bound_data has {n} entries, sim_data has {len(sim_data)}")
    
    # Initialize trajectory arrays
    time_array = np.zeros(n)
    mass_traj = np.zeros(n)
    energy_traj = np.zeros(n)
    r_traj = np.zeros((n, 3))
    d_traj = np.zeros(n)
    v_traj = np.zeros((n, 3))
    
    # Extract data for each timestep
    for i in range(n): 
        # Extract bound particle properties
        mass_traj[i] = bound_data[i]["total mass"]
        energy_traj[i] = bound_data[i]["total energy"]
        r_traj[i, :] = bound_data[i]["pos"]
        v_traj[i, :] = bound_data[i]["vel"]
        d_traj[i] = np.linalg.norm(r_traj[i,:]) 
        
        # Extract time from simulation data
        time_array[i] = sim_data[i]["time"] 
    
    # Package results into dictionary
    traj = {
        'mass': mass_traj,
        'energy': energy_traj,
        'time': time_array,
        'pos': r_traj,
        'vel': v_traj,
        'distance': d_traj
    }
    
    return traj

if not loaded_from_cache:
    traj = trajectories(bound_data, sim_data)

### NEW: binary outputs for the 1 Gyr collisionless run

These tables keep the NGC6569 run products in raw binary form, with JSON sidecars recording the columns and shapes. The PeTar-start table is just mass plus phase space in the GC frame.


In [ ]:
if not loaded_from_cache:
    initial_cluster_table = np.column_stack((mass, xv, r_agama, v_agama))
    initial_gc_table = np.column_stack((mass, pos_0, vel_0))
    final_gc_table = np.column_stack((mass, sim_data[-1]["pos"], sim_data[-1]["vel"]))
    mass_loss_table = np.column_stack((
        traj["time"] * 1000.0 + tfin.value,
        traj["time"],
        traj["mass"],
        traj["mass"] / traj["mass"][0],
        traj["energy"],
        traj["distance"],
    ))
    center_orbit_table = np.column_stack((
        traj["time"] * 1000.0 + tfin.value,
        traj["time"],
        traj["pos"],
        traj["vel"],
        traj["distance"],
    ))

    write_binary_table(
        OUT / "ngc6569_initial_cluster_frame.bin",
        initial_cluster_table,
        columns=["mass", "x", "y", "z", "vx", "vy", "vz", "r", "speed"],
        units={"mass": "Msun", "x": "kpc", "y": "kpc", "z": "kpc", "vx": "km/s", "vy": "km/s", "vz": "km/s", "r": "kpc", "speed": "km/s"},
        description="Initial King-model particles before shifting to the NGC6569 orbit.",
    )
    write_binary_table(
        OUT / "ngc6569_petar_initial.bin",
        initial_gc_table,
        columns=["mass", "x_gc", "y_gc", "z_gc", "vx_gc", "vy_gc", "vz_gc"],
        units={"mass": "Msun", "x_gc": "kpc", "y_gc": "kpc", "z_gc": "kpc", "vx_gc": "km/s", "vy_gc": "km/s", "vz_gc": "km/s"},
        description="PeTar-style initial conditions in the Galactocentric frame.",
    )
    write_binary_table(
        OUT / "ngc6569_final_particles.bin",
        final_gc_table,
        columns=["mass", "x_gc", "y_gc", "z_gc", "vx_gc", "vy_gc", "vz_gc"],
        units={"mass": "Msun", "x_gc": "kpc", "y_gc": "kpc", "z_gc": "kpc", "vx_gc": "km/s", "vy_gc": "km/s", "vz_gc": "km/s"},
        description="Final particle snapshot from the 1 Gyr collisionless run.",
    )
    write_binary_table(
        OUT / "ngc6569_mass_loss.bin",
        mass_loss_table,
        columns=["time_to_present", "elapsed_time", "bound_mass", "bound_mass_fraction", "bound_energy", "galactocentric_distance"],
        units={"time_to_present": "Myr", "elapsed_time": "Gyr", "bound_mass": "Msun", "galactocentric_distance": "kpc"},
        description="Mass-loss history for the 1 Gyr NGC6569 collisionless run.",
    )
    write_binary_table(
        OUT / "ngc6569_center_orbit.bin",
        center_orbit_table,
        columns=["time_to_present", "elapsed_time", "x_gc", "y_gc", "z_gc", "vx_gc", "vy_gc", "vz_gc", "galactocentric_distance"],
        units={"time_to_present": "Myr", "elapsed_time": "Gyr", "x_gc": "kpc", "y_gc": "kpc", "z_gc": "kpc", "vx_gc": "km/s", "vy_gc": "km/s", "vz_gc": "km/s", "galactocentric_distance": "kpc"},
        description="Center-of-mass orbit recovered from bound particles.",
    )
    write_manifest(
        OUT / "ngc6569_run_manifest.json",
        {
            "source_notebook": "NGC6569_Dev3_original.ipynb",
            "run_notebook": "NGC6569_Dev3_original_binary_1Gyr.ipynb",
            "duration_Myr": float(abs(tfin.value)),
            "Nbody": int(len(mass)),
            "cache_key": ngc6569_cache_key,
            "cache_payload": ngc6569_cache_payload,
            "binary_tables": [
                "ngc6569_initial_cluster_frame.bin",
                "ngc6569_petar_initial.bin",
                "ngc6569_final_particles.bin",
                "ngc6569_mass_loss.bin",
                "ngc6569_center_orbit.bin",
            ],
        },
    )
else:
    print("Using cached binary run products; not rewriting them.")


In [ ]:
def get_tidal_data(traj, pot_ext): 

    pos = traj["pos"]
    n=pos.shape[0]

    tidal = np.zeros((n,3))
    max_tidal = np.zeros(n)

    for i in range(n): 

        H = np.zeros((3, 3))
        
        hess = pot_ext.eval(pos[i,:], der=True)
    
    
        # Upper triangular indices
        triu_indices = np.triu_indices(3)
        H[triu_indices] = hess
    
        # Make symmetric
        H = H + H.T - np.diag(np.diag(H))

        tidal[i,:] = np.linalg.eigvals(H)

        max_tidal[i] = np.max(tidal[i,:]) 

    return tidal, max_tidal 

tidal, max_tidal = get_tidal_data(traj, pot_ext)

In [ ]:

max_t=np.max(max_tidal)

fig = plt.figure(figsize=(14, 8 ))
fig.suptitle('Mass Loss, Energy, Max Tidal Force, Radius ', size=20)
ax = fig.add_subplot(411)
ax.set_xlim(tfin.value, 0)
#ax.set_xlim(tfin.value, -900)
ax.set_ylabel(r"$M/M_0$", size=20) 
#ax.set_xlabel("Myr", size=20)

M0=traj["mass"][0]
ax.plot(time_array+tfin.value, traj["mass"]/M0)


ax = fig.add_subplot(412)
ax.set_xlim(tfin.value, 0)
#ax.set_xlim(tfin.value, -900)
ax.set_ylabel(r"$E/|E_0|$", size=20) 
#ax.set_xlabel("Myr", size=20) 
E = traj["energy"]
ax.plot(time_array+tfin.value, E/np.abs(E[0])) 


ax = fig.add_subplot(413)
ax.set_xlim(tfin.value, 0)
#ax.set_xlim(tfin.value, -900)
ax.set_ylabel(r"$\lambda_{max}$ ", size=20) 
ax.set_xlabel("Myr", size=20) 
ax.plot(time_array + tfin.value, max_tidal)


ax = fig.add_subplot(414)
ax.set_xlim(tfin.value, 0)
#ax.set_xlim(tfin.value, -900)
ax.set_ylabel(r"$r(t)$ [kpc]", size=20) 
ax.set_xlabel("Myr", size=20) 
ax.plot(time_array + tfin.value, traj["distance"])

fig.savefig("6569_ml_II.pdf") 

# print("Maybe add distance from galactic center." ) 
# print("How does tidal stripping compare at large tidal tensor versus Particle Spray method.") 
# print("why is there a dip in energy right before an increase?" ) 


### NEW: save the 1 Gyr mass-loss figure beside the binary data


In [ ]:
fig.savefig(OUT / "ngc6569_1gyr_mass_loss_history.pdf")


## NEW: Chen-validation Step 2 style cached sweep

This is the same idea as Step 2 in `NGC6569_chen_validation.ipynb`: run parameter curves through one cached function. Here the cache is binary rather than NumPy archive. Leave `RUN_NGC6569_STEP2 = False` unless you want to launch the optional sweep.


In [ ]:
STEP2_CACHE = OUT / "cache"
STEP2_CACHE.mkdir(parents=True, exist_ok=True)

def _step2_cache_path(payload):
    h = binary_cache_key(payload)
    stem = (
        f"ml_W{payload['W0']:g}_r{payload['scale_radius_kpc']:.5f}_"
        f"k{payload['kmax']}_eps{payload['eps_pc']:g}_"
        f"m{payload['m_star']:g}_N{payload['Nbody']}_{h}"
    )
    return STEP2_CACHE / f"{stem}.bin"

def run_ngc6569_mass_loss(W0, scale_radius, kmax, eps_pc, m_star, use_cache=True):
    Nbody_run = int(round(m / m_star))
    payload = {
        "W0": float(W0),
        "scale_radius_kpc": float(scale_radius),
        "kmax": int(kmax),
        "eps_pc": float(eps_pc),
        "m_star": float(m_star),
        "Nbody": int(Nbody_run),
        "cluster_mass_Msun": float(m),
        "tfin_Myr": float(tfin.to(u.Myr).value),
        "downsample": int(downsample),
        "potential": "MWPotentialHunter24_rotating.ini",
    }
    cache_path = _step2_cache_path(payload)
    if use_cache and binary_table_ready(cache_path):
        table, _ = read_binary_table(cache_path)
        return table[:, 0], table[:, 1]

    xv_run, mass_run, _, _ = sample_king_model(W0, scale_radius, m, Nbody_run)
    pos_run, vel_run = xv_to_xyz_vel(xv_run, center_pos_0, center_vel_0)
    tau_run = 2 ** (-kmax) * time_unit
    nt_run = int(tmax / tau_run) + 1
    eps_run = eps_pc / 1000.0

    sim_run = kdk_leapfrog(pot_use, pos_run, vel_run, mass_run, nt_run, tau_run,
                           agama.G, eps_run, time_unit, downsample, last_snapshot=False)
    bound_run = get_bound_particles(sim_run, mass_run)
    traj_run = trajectories(bound_run, sim_run)
    t_myr = traj_run["time"] * 1000.0
    M_over_M0 = traj_run["mass"] / traj_run["mass"][0]
    table = np.column_stack((t_myr, M_over_M0))

    if use_cache:
        write_binary_table(
            cache_path,
            table,
            columns=["elapsed_time", "bound_mass_fraction"],
            units={"elapsed_time": "Myr"},
            description="Cached NGC6569 mass-loss curve for the Chen Step 2 style sweep.",
        )
        meta = json.loads(cache_path.with_suffix(".json").read_text())
        meta["cache_key"] = binary_cache_key(payload)
        meta["cache_payload"] = payload
        write_manifest(cache_path.with_suffix(".json"), meta)
    return t_myr, M_over_M0


### NEW: optional Step 2 sweep

Set `RUN_NGC6569_STEP2 = True` to run the cached W, timestep, softening, and particle-mass comparisons.


In [ ]:
RUN_NGC6569_STEP2 = False

if RUN_NGC6569_STEP2:
    import matplotlib.cm as cm
    from matplotlib.colors import Normalize

    fid = {
        "W0": W0_value,
        "scale_radius": r_scale,
        "kmax": kmax,
        "eps_pc": eps * 1000.0,
        "m_star": m / Nbody,
    }
    results = {"W": {}, "tau": {}, "eps": {}, "m": {}}
    for W in [2, 4, 6, 8, 10, 12, 14]:
        print("W", W)
        results["W"][W] = run_ngc6569_mass_loss(W, fid["scale_radius"], fid["kmax"], fid["eps_pc"], fid["m_star"])
    for k in [10, 11, 12, 13, 14, 15]:
        print("kmax", k)
        results["tau"][k] = run_ngc6569_mass_loss(fid["W0"], fid["scale_radius"], k, fid["eps_pc"], fid["m_star"])
    for e in [2**-2, 2**-1, 2**0, 2**1, 2**2]:
        print("eps", e)
        results["eps"][e] = run_ngc6569_mass_loss(fid["W0"], fid["scale_radius"], fid["kmax"], e, fid["m_star"])
    for ms in [2**-3 * 10, 2**0 * 10, 2**3 * 10]:
        print("m", ms)
        results["m"][ms] = run_ngc6569_mass_loss(fid["W0"], fid["scale_radius"], fid["kmax"], fid["eps_pc"], ms)
    print("done")

    def _colors(values, fiducial):
        vals = sorted(values)
        norm = Normalize(vmin=min(vals), vmax=max(vals))
        return {v: ("black" if np.isclose(v, fiducial) else cm.coolwarm(norm(v))) for v in vals}

    fig, axes = plt.subplots(4, 1, figsize=(6, 14), sharex=True)
    fig.suptitle("NGC6569 Chen Step 2 style sweep", style="italic", size=16)

    ax = axes[0]; cols = _colors(results["W"], fid["W0"])
    for W in sorted(results["W"]):
        t, Mfrac = results["W"][W]
        ax.plot(t, Mfrac, color=cols[W], lw=1.5, label=fr"$W = {int(W)}$")

    ax = axes[1]; cols = _colors(results["tau"], fid["kmax"])
    for k in sorted(results["tau"]):
        t, Mfrac = results["tau"][k]
        ax.plot(t, Mfrac, color=cols[k], lw=1.5, label=fr"$\tau = 2^{{-{k}}}$")

    ax = axes[2]; cols = _colors(results["eps"], fid["eps_pc"])
    for e in sorted(results["eps"]):
        t, Mfrac = results["eps"][e]
        ax.plot(t, Mfrac, color=cols[e], lw=1.5, label=fr"$\epsilon = {e:g}$ pc")

    ax = axes[3]; cols = _colors(results["m"], fid["m_star"])
    for ms in sorted(results["m"]):
        t, Mfrac = results["m"][ms]
        ax.plot(t, Mfrac, color=cols[ms], lw=1.5, label=fr"$m = {ms:g}\ M_\odot$")

    for ax in axes:
        ax.set_xlim(0, abs(tfin.to(u.Myr).value))
        ax.set_ylabel(r"$M(t)/M_0$", size=14)
        ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=10)
    axes[-1].set_xlabel(r"$t$ [Myr]", size=14)
    plt.tight_layout()
    fig.savefig(OUT / "ngc6569_step2_chen_style_massloss.pdf", bbox_inches="tight")
else:
    print("Set RUN_NGC6569_STEP2 = True to run the optional cached sweep.")
